# LeetCode #139 — Word Break: DP trên tiền tố với từ điển dạng set

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, String, Dynamic Programming, Trie, Memoization

Skeleton trên LeetCode:

```python
class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> bool:
```

Mục tiêu của notebook:

1. Thấy vì sao **tham lam** (lấy từ dài nhất / ngắn nhất trước) sai.
2. Đệ quy thử mọi cách cắt — và vì sao nó bùng nổ trên `"aaaa…ab"`.
3. DP `dp[i]` = "tiền tố `s[:i]` có tách được không" với hai cách duyệt: theo **mọi điểm cắt** và theo **mọi từ**.
4. Tối ưu: giới hạn độ dài từ, và BFS trên các vị trí.

# 1. Bài toán là gì?

Cho chuỗi `s` và từ điển `wordDict`. Trả về `True` nếu `s` có thể tách thành **dãy một hoặc nhiều** từ trong từ điển (mỗi từ được **dùng lại** nhiều lần).

**Constraints (theo đề):**

- `1 <= s.length <= 300`
- `1 <= wordDict.length <= 1000`
- `1 <= wordDict[i].length <= 20`
- `s`, `wordDict[i]` gồm chữ cái tiếng Anh viết thường; các từ trong từ điển **phân biệt**.

---

## Ví dụ

```text
s = "leetcode",      wordDict = ["leet", "code"]                    ->  True
s = "applepenapple", wordDict = ["apple", "pen"]                    ->  True
s = "catsandog",     wordDict = ["cats", "dog", "sand", "and", "cat"]  ->  False
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. Bảng băm (hash table) — `dict` và `set`
3. Chuỗi (string) trong Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [3]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [4]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 543.05 ms | trên set: 0.075 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [5]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [6]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [7]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [8]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [9]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [10]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.4 Áp dụng vào bài này

**DP trên tiền tố**: `dp[i]` = `s[:i]` tách được thành các từ trong từ điển không; `dp[i]` đúng nếu có `j < i` với `dp[j]` đúng và `s[j:i]` thuộc từ điển. Từ điển chuyển thành **set** để tra `O(1)`.

# 3. Tham lam có đúng không?

Thử lấy **từ dài nhất** khớp ở đầu chuỗi, rồi lặp lại:

In [11]:
from typing import List
from functools import lru_cache
from collections import deque
import random, time

def greedy_longest(s: str, words: List[str]) -> bool:
    i = 0
    while i < len(s):
        best = max((w for w in words if s.startswith(w, i)), key=len, default=None)
        if best is None:
            return False
        i += len(best)
    return True

print(greedy_longest("leetcode", ["leet", "code"]), "(đúng: True)")

True (đúng: True)


In [12]:
s, words = "abcd", ["abc", "ab", "cd"]
print("Tham lam dài nhất cho 'abcd' với", words, "->", greedy_longest(s, words), "(đúng: True = 'ab' + 'cd')")

Tham lam dài nhất cho 'abcd' với ['abc', 'ab', 'cd'] -> False (đúng: True = 'ab' + 'cd')


Tham lam lấy `"abc"` (dài nhất) rồi kẹt ở `"d"`, trong khi `"ab" + "cd"` mới đúng. Lấy từ **ngắn** nhất trước cũng có phản ví dụ tương tự. ⇒ Cần thử mọi lựa chọn → đệ quy / DP.

# 4. Baseline: đệ quy thử mọi từ ở mỗi vị trí

`can(i)` = `s[i:]` tách được không. Thử mọi từ `w` khớp tại `i`, đi tiếp `can(i + len(w))`.

In [13]:
def word_break_rec(s: str, words: List[str]):
    calls = 0
    def can(i):
        nonlocal calls
        calls += 1
        if i == len(s):
            return True
        return any(s.startswith(w, i) and can(i + len(w)) for w in words)
    return can(0), calls

print(word_break_rec("catsandog", ["cats", "dog", "sand", "and", "cat"]))
print(word_break_rec("a" * 22 + "b", ["a", "aa", "aaa", "aaaa"]))

(False, 5)


(False, 2192444)


Chuỗi `"aaa…ab"` với từ điển `a, aa, aaa, aaaa`: mọi cách tách phần `"aaa…a"` đều được thử rồi thất bại ở `"b"` — số lời gọi tăng **hàm mũ**. Nhưng chỉ có `n + 1` vị trí `i` khác nhau → memo / DP.

# 5. Ý tưởng chính: DP trên tiền tố

`dp[i] = True` nếu `s[:i]` tách được. `dp[0] = True` (chuỗi rỗng).

```text
dp[i] = True  nếu tồn tại j < i với dp[j] = True và s[j:i] ∈ word_set
```

Đáp án `dp[n]`. Dùng `set` để kiểm tra `s[j:i] ∈ word_set` trung bình `O(độ dài)`.

In [14]:
def word_break_dp(s: str, words: List[str]) -> bool:
    word_set = set(words)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break
    return dp[n]

for s, w in [("leetcode", ["leet", "code"]), ("applepenapple", ["apple", "pen"]), ("catsandog", ["cats", "dog", "sand", "and", "cat"])]:
    print(s, word_break_dp(s, w))

leetcode True
applepenapple True
catsandog False


# 6. Mô phỏng

In [15]:
def trace_dp(s, words):
    word_set = set(words)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                print(f"dp[{i}] = True vì dp[{j}] = True và s[{j}:{i}] = {s[j:i]!r} có trong từ điển")
                break
    print("dp =", "".join("T" if x else "." for x in dp))
    print("Kết quả:", dp[n])
    return dp[n]

_ = trace_dp("catsandog", ["cats", "dog", "sand", "and", "cat"])

dp[3] = True vì dp[0] = True và s[0:3] = 'cat' có trong từ điển
dp[4] = True vì dp[0] = True và s[0:4] = 'cats' có trong từ điển
dp[7] = True vì dp[3] = True và s[3:7] = 'sand' có trong từ điển
dp = T..TT..T..
Kết quả: False


`"catsandog"`: tới được vị trí 3 (`cat`), 4 (`cats`), 7 (`cat`+`sand` hoặc `cats`+`and`), nhưng `s[7:9] = "og"` không phải từ → `dp[9] = False`.

# 7. Tối ưu: duyệt theo từ, hoặc giới hạn độ dài

Mỗi từ dài tối đa 20 → với mỗi `i`, chỉ cần thử `j` trong `[i − 20, i)`. Hoặc **duyệt theo từ**: với mỗi `i` có `dp[i] = True`, đánh dấu `dp[i + len(w)]` cho mọi `w` khớp tại `i`.

In [16]:
def word_break_by_word(s: str, words: List[str]) -> bool:
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(n):
        if not dp[i]:
            continue
        for w in words:
            if s.startswith(w, i):
                dp[i + len(w)] = True
    return dp[n]


def word_break_maxlen(s: str, words: List[str]) -> bool:
    word_set = set(words)
    max_len = max(map(len, words))
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(max(0, i - max_len), i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break
    return dp[n]

print(word_break_by_word("catsandog", ["cats", "dog", "sand", "and", "cat"]), word_break_maxlen("leetcode", ["leet", "code"]))

False True


| Cách | Time |
|---|---|
| Mọi `j < i` | `O(n² · L)` (L = độ dài lát cắt khi băm) |
| Giới hạn `max_len` | `O(n · max_len²)` |
| Theo từ | `O(n · W · max_len)` (W = số từ) |

Chọn cách theo kích thước: từ điển nhỏ → theo từ; từ điển lớn, từ ngắn → giới hạn `max_len`.

# 8. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> bool:
        word_set = set(wordDict)
        max_len = max(map(len, wordDict))
        n = len(s)
        dp = [False] * (n + 1)
        dp[0] = True
        for i in range(1, n + 1):
            for j in range(max(0, i - max_len), i):
                if dp[j] and s[j:i] in word_set:
                    dp[i] = True
                    break
        return dp[n]

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def brute(s, words):
    @lru_cache(maxsize=None)
    def can(i):
        return i == len(s) or any(s.startswith(w, i) and can(i + len(w)) for w in words)
    return can(0)


sol = Solution()
check("Ví dụ 1", sol.wordBreak("leetcode", ["leet", "code"]), True)
check("Ví dụ 2", sol.wordBreak("applepenapple", ["apple", "pen"]), True)
check("Ví dụ 3", sol.wordBreak("catsandog", ["cats", "dog", "sand", "and", "cat"]), False)
check("Tham lam bị lừa", sol.wordBreak("abcd", ["abc", "ab", "cd"]), True)
check("Bùng nổ đệ quy", sol.wordBreak("a" * 299 + "b", ["a", "aa", "aaa", "aaaa"]), False)

random.seed(139)
for _ in range(3000):
    words = list({"".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 4))})
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 12)))
    expected = brute(s, tuple(words))
    assert sol.wordBreak(s, words) == expected == word_break_dp(s, words) == word_break_by_word(s, words)
print("3000 test ngẫu nhiên: 3 cách DP khớp đệ quy có memo")

Ví dụ 1            PASS  True
Ví dụ 2            PASS  True
Ví dụ 3            PASS  False
Tham lam bị lừa    PASS  True
Bùng nổ đệ quy     PASS  False
3000 test ngẫu nhiên: 3 cách DP khớp đệ quy có memo


# 9. Vì sao đúng? Độ phức tạp?

Một cách tách `s[:i]` (với `i > 0`) có **từ cuối** `s[j:i]` cho một `j < i`, và phần trước `s[:j]` phải tách được. Ngược lại, nếu `dp[j]` và `s[j:i]` là từ thì ghép lại được cách tách `s[:i]`. Vậy công thức bao trùm đúng mọi trường hợp; `dp[0] = True` là cách tách rỗng.

```text
Time:  O(n · max_len · max_len)  (bản LeetCode: lát cắt dài ≤ max_len, băm O(max_len))
Space: O(n + tổng độ dài từ điển)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Tham lam

Như mục 3.

## Lỗi 2: `dp[0] = False`

Không có điểm xuất phát → mọi `dp[i]` đều `False`.

In [19]:
def wrong_dp0(s, words):
    ws, n = set(words), len(s)
    dp = [False] * (n + 1)                         # SAI: thiếu dp[0] = True
    for i in range(1, n + 1):
        dp[i] = any(dp[j] and s[j:i] in ws for j in range(i))
    return dp[n]

print("Sai :", wrong_dp0("leetcode", ["leet", "code"]), "| Đúng:", sol.wordBreak("leetcode", ["leet", "code"]))

Sai : False | Đúng: True


## Lỗi 3: Kiểm tra `s[j:i] in wordDict` với `wordDict` là **list**

Đúng nhưng mỗi lần kiểm tra tốn `O(W)` → chậm khi từ điển 1000 từ. Đổi sang `set`.

In [20]:
def dp_generic(s, container):
    n = len(s)
    dp = [True] + [False] * n
    for i in range(1, n + 1):
        dp[i] = any(dp[j] and s[j:i] in container for j in range(i))
    return dp[n]

words = ["w%03d" % i for i in range(1000)] + ["a", "b"]
s = "ab" * 100
t0 = time.time(); r1 = dp_generic(s, set(words)); t1 = time.time()
r2 = dp_generic(s, words); t2 = time.time()
print(f"set: {t1 - t0:.3f}s | list: {t2 - t1:.3f}s | kết quả giống: {r1 == r2}")

set: 0.003s | list: 0.191s | kết quả giống: True

## Lỗi 4: Đệ quy không memo

Như mục 4 — bùng nổ hàm mũ trên `"aaa…ab"`.

# 11. Mẹo nhận diện pattern

**"Tách chuỗi thành các mảnh thuộc một tập cho trước"** → DP trên tiền tố, `dp[i]` dựa vào `dp[j]` và mảnh `s[j:i]`.

| Bài | Hỏi | Thay đổi |
|---|---|---|
| #139 Word Break | có tách được không | `or` |
| #140 Word Break II (Hard) | liệt kê mọi cách | backtracking + memo |
| #91 Decode Ways | đếm số cách | `+`, mảnh dài 1–2 |
| #472 Concatenated Words (Hard) | từ nào ghép được từ các từ khác | #139 cho từng từ |
| #279 Perfect Squares | số mảnh ít nhất | `min` |

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `dp` cho `s = "pineapplepenapple"`, `wordDict = ["apple", "pen", "applepen", "pine", "pineapple"]`.

## Bài 2 — Điền chỗ trống

In [21]:
def exercise_2_word_break(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        # TODO
        pass
    return dp[n]


# assert exercise_2_word_break("leetcode", ["leet", "code"]) is True
# assert exercise_2_word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False

## Bài 3 — Biến thể: số từ ít nhất

Trả về số từ ít nhất để tách `s`, hoặc `-1` nếu không thể.

```python
min_words("applepenapple", ["apple", "pen", "applepen"]) -> 2     # "applepen" + "apple"
```

In [22]:
def exercise_3_min_words(s: str, words: List[str]) -> int:
    # TODO
    pass


# assert exercise_3_min_words("applepenapple", ["apple", "pen", "applepen"]) == 2
# assert exercise_3_min_words("catsandog", ["cats", "dog", "sand", "and", "cat"]) == -1
# assert exercise_3_min_words("aaaa", ["a", "aa"]) == 2

## Bài 4 — Biến thể: Word Break II (LeetCode #140)

Liệt kê mọi cách tách, mỗi cách là các từ nối bằng dấu cách.

```python
word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"]) -> ["cats and dog", "cat sand dog"]
```

In [23]:
def exercise_4_word_break_ii(s: str, words: List[str]) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_4_word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"])) == ["cat sand dog", "cats and dog"]
# assert exercise_4_word_break_ii("catsandog", ["cats", "dog", "sand", "and", "cat"]) == []

## Bài 5 — Mini challenge: Concatenated Words (LeetCode #472, Hard)

Cho danh sách từ (phân biệt), trả về các từ có thể ghép từ **ít nhất hai** từ **khác** trong danh sách.

```python
concatenated(["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"])
  -> ["catsdogcats", "dogcatsdog", "ratcatdogcat"]
```

Gợi ý nhanh: sắp xếp theo độ dài; với mỗi từ, chạy Word Break với tập các từ **ngắn hơn** đã xét.

In [24]:
def exercise_5_concatenated(words: List[str]) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_5_concatenated(["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"])) == ["catsdogcats", "dogcatsdog", "ratcatdogcat"]
# assert exercise_5_concatenated(["cat", "dog", "catdog"]) == ["catdog"]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_dp` sau khi tự làm.

**Bài 3**
- Mức 1: `dp[i]` = số từ ít nhất cho `s[:i]`, `dp[0] = 0`, còn lại `∞`; `dp[i] = min(dp[j] + 1)`.

**Bài 4**
- Mức 1: `go(i)` (có memo) trả list các câu cho `s[i:]`; `go(n) = [""]`.
- Mức 2: với mỗi từ khớp tại `i`, ghép `w + (" " + rest if rest else "")`.

**Bài 5**
- Mức 1: duyệt từ theo độ dài tăng dần; giữ `seen` là set các từ đã duyệt.
- Mức 2: nếu từ hiện tại (không rỗng) Word Break được với `seen` thì thêm vào kết quả; rồi thêm vào `seen`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
_ = trace_dp("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"])

dp[4] = True vì dp[0] = True và s[0:4] = 'pine' có trong từ điển
dp[9] = True vì dp[0] = True và s[0:9] = 'pineapple' có trong từ điển
dp[12] = True vì dp[4] = True và s[4:12] = 'applepen' có trong từ điển
dp[17] = True vì dp[12] = True và s[12:17] = 'apple' có trong từ điển
dp = T...T....T..T....T
Kết quả: True


## Đáp án Bài 2

In [26]:
def answer_2_word_break(s: str, wordDict: List[str]) -> bool:
    return word_break_dp(s, wordDict)

assert answer_2_word_break("leetcode", ["leet", "code"]) is True
assert answer_2_word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False
print("Bài 2 PASS (giống word_break_dp ở mục 5)")

Bài 2 PASS (giống word_break_dp ở mục 5)


## Đáp án Bài 3

In [27]:
def answer_3_min_words(s: str, words: List[str]) -> int:
    ws, n = set(words), len(s)
    INF = float("inf")
    dp = [0] + [INF] * n
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] + 1 < dp[i] and s[j:i] in ws:
                dp[i] = dp[j] + 1
    return dp[n] if dp[n] < INF else -1

assert answer_3_min_words("applepenapple", ["apple", "pen", "applepen"]) == 2
assert answer_3_min_words("catsandog", ["cats", "dog", "sand", "and", "cat"]) == -1
assert answer_3_min_words("aaaa", ["a", "aa"]) == 2
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_word_break_ii(s: str, words: List[str]) -> List[str]:
    @lru_cache(maxsize=None)
    def go(i):
        if i == len(s):
            return [""]
        out = []
        for w in words:
            if s.startswith(w, i):
                for rest in go(i + len(w)):
                    out.append(w + (" " + rest if rest else ""))
        return out
    return go(0)

assert sorted(answer_4_word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"])) == ["cat sand dog", "cats and dog"]
assert answer_4_word_break_ii("catsandog", ["cats", "dog", "sand", "and", "cat"]) == []
for _ in range(1000):
    words = list({"".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 4))})
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 8)))
    res = answer_4_word_break_ii(s, words)
    assert (len(res) > 0) == sol.wordBreak(s, words)
    assert all("".join(r.split(" ")) == s and all(p in words for p in r.split(" ")) for r in res)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [29]:
def answer_5_concatenated(words: List[str]) -> List[str]:
    seen, out = set(), []
    for w in sorted(words, key=len):
        if w and seen:
            n = len(w)
            dp = [True] + [False] * n
            for i in range(1, n + 1):
                dp[i] = any(dp[j] and w[j:i] in seen for j in range(i))
            if dp[n]:
                out.append(w)
        seen.add(w)
    return out

assert sorted(answer_5_concatenated(["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"])) == ["catsdogcats", "dogcatsdog", "ratcatdogcat"]
assert answer_5_concatenated(["cat", "dog", "catdog"]) == ["catdog"]
print("Bài 5 PASS")

Bài 5 PASS


Vì chỉ dùng các từ **ngắn hơn** (đã nằm trong `seen`), một từ không thể "ghép" từ chính nó — đảm bảo điều kiện "ít nhất hai từ khác".

# 15. Tổng kết

> `dp[0] = True`; `dp[i] = True` nếu có `j < i` với `dp[j]` và `s[j:i] ∈ set(wordDict)`. Giới hạn `j ≥ i − max_len` để nhanh hơn. Tham lam và đệ quy không memo đều thất bại.

---

## Checklist trước khi nộp

- [ ] `dp[0] = True`?
- [ ] Từ điển chuyển sang `set`?
- [ ] Không dùng tham lam?
- [ ] Có memo / DP (không đệ quy trần)?